<a href="https://colab.research.google.com/github/fajriantomanungki/lomba_kti/blob/main/Data/SPBU_Density.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install osmnx geopandas pyogrio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.0 MB/s eta 0:00:00


In [1]:
!apt-get install osmium-tool -qq

Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../libboost-program-options1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package osmium-tool.
Preparing to unpack .../osmium-tool_1.14.0-1_amd64.deb ...
Unpacking osmium-tool (1.14.0-1) ...
Setting up libboost-program-options1.74.0:amd64 (1.74.0-14ubuntu3) ...
Setting up osmium-tool (1.14.0-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.s

In [3]:
import os

folder = "/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/data"
pbf_path = f"{folder}/indonesia.osm.pbf"

if not os.path.exists(pbf_path):
    !wget -O "{pbf_path}" https://download.geofabrik.de/asia/indonesia-latest.osm.pbf
else:
    print("File sudah ada:", pbf_path)

File sudah ada: /content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/data/indonesia.osm.pbf


In [4]:
!osmium tags-filter "{pbf_path}" nwr/amenity=fuel \
-o "{folder}/spbu_indonesia.osm.pbf" --overwrite

[======================================================================] 100% 


In [5]:
!osmium export "{folder}/spbu_indonesia.osm.pbf" \
-o "{folder}/spbu_indonesia.geojson" --overwrite

[======================================================================] 100% 


In [7]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file(f"{folder}/batas_kabkot_sulampua.geojson").to_crs(epsg=4326)

spbu_all = gpd.read_file(f"{folder}/spbu_indonesia.geojson").to_crs(epsg=4326)

print(spbu_all.shape)
spbu_all.head()

(7714, 226)


,Commercial,M:flood_prone,M:hazard_prone,PFM:garmin_type,access,access:roof,addr:city,addr:country,addr:district,addr:full,...,wall,water_source:id,water_supply,website,wheelchair,wheelchair:description,wikidata,wikipedia,wpt_description,geometry
0,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (107.60203 -6.88768)
1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (107.58585 -6.907)
2,None,None,None,None,None,None,Bandung,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (107.63432 -6.89973)
3,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (107.59871 -6.8806)
4,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,POINT (107.58406 -6.91859)


In [8]:
spbu_sulampua = gpd.sjoin(
    spbu_all,
    gdf[["NAME_1", "NAME_2", "geometry"]],
    how="inner",
    predicate="within"
)

In [9]:
spbu_count = (
    spbu_sulampua
    .groupby(["NAME_1", "NAME_2"])
    .size()
    .reset_index(name="jumlah_spbu")
)

result = gdf.merge(
    spbu_count,
    on=["NAME_1", "NAME_2"],
    how="left"
)

result["jumlah_spbu"] = result["jumlah_spbu"].fillna(0)

result_proj = result.to_crs(epsg=3857)
result_proj["luas_km2"] = result_proj.area / 1e6
result_proj["spbu_density"] = (
    result_proj["jumlah_spbu"] / result_proj["luas_km2"]
) * 1000

In [10]:
final_spbu_density = result_proj[[
    "NAME_1",
    "NAME_2",
    "jumlah_spbu",
    "luas_km2",
    "spbu_density"
]].copy()

final_spbu_density["luas_km2"] = final_spbu_density["luas_km2"].round(2)
final_spbu_density["spbu_density"] = final_spbu_density["spbu_density"].round(4)

final_spbu_density.sort_values("spbu_density", ascending=False)

,NAME_1,NAME_2,jumlah_spbu,luas_km2,spbu_density
83,SulawesiSelatan,Makassar,117.0,184.85,632.9359
5,Gorontalo,KotaGorontalo,17.0,65.58,259.2056
127,SulawesiUtara,Manado,32.0,164.27,194.7986
87,SulawesiSelatan,Parepare,16.0,88.56,180.6730
111,SulawesiTenggara,Kendari,34.0,272.04,124.9821
...,...,...,...,...,...
121,SulawesiUtara,BolaangMongondowSelatan,0.0,1815.44,0.0000
119,SulawesiUtara,Bitung,0.0,320.27,0.0000
118,SulawesiTenggara,Wakatobi,0.0,497.22,0.0000
126,SulawesiUtara,Kotamobagu,0.0,51.00,0.0000


In [11]:
folder = "/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/data"

csv_path = f"{folder}/spbu_density_sulampua.csv"

final_spbu_density.to_csv(
    csv_path,
    index=False
)

print("CSV tersimpan:")
print(csv_path)

CSV tersimpan:
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/data/spbu_density_sulampua.csv


In [12]:
geojson_path = f"{folder}/spbu_density_sulampua.geojson"

result_proj.to_crs(epsg=4326).to_file(
    geojson_path,
    driver="GeoJSON"
)

print("GeoJSON tersimpan:")
print(geojson_path)

GeoJSON tersimpan:
/content/drive/MyDrive/Karya Ilmiah/AEDS_Sulampua/data/spbu_density_sulampua.geojson


In [27]:
final_spbu_density[
    final_spbu_density["jumlah_spbu"] == 0
][["NAME_1", "NAME_2"]]

,NAME_1,NAME_2
0,Gorontalo,Boalemo
2,Gorontalo,DanauLimboto
6,Gorontalo,Pohuwato
8,Maluku,Buru
10,Maluku,KepulauanAru
11,Maluku,MalukuBaratDaya
13,Maluku,MalukuTenggara
14,Maluku,MalukuTenggaraBarat
15,Maluku,SeramBagianBarat
16,Maluku,SeramBagianTimur


In [22]:
!pip install duckduckgo-search beautifulsoup4 lxml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 53.7 MB/s eta 0:00:00


In [23]:
from duckduckgo_search import DDGS
import pandas as pd
import time

In [26]:
zero_area = final_spbu_density[
    final_spbu_density["jumlah_spbu"] == 0
][["NAME_1", "NAME_2"]]

In [29]:
validation_result = []

with DDGS() as ddgs:

    for _, row in zero_area.iterrows():

        prov = row["NAME_1"]
        kab = row["NAME_2"]

        query = f"SPBU Pertamina di {kab} {prov}"

        print(query)

        try:
            results = list(
                ddgs.text(
                    query,
                    max_results=10
                )
            )

            count = len(results)

            validation_result.append({
                "NAME_1": prov,
                "NAME_2": kab,
                "web_result_count": count
            })

            print("hasil:", count)

            time.sleep(2)

        except Exception as e:
            print(e)

/tmp/ipykernel_15553/1027149827.py:3: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


SPBU Pertamina di Boalemo Gorontalo


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


hasil: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

SPBU Pertamina di SiauTagulandangBiaro SulawesiUtara


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

hasil: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [31]:
validation_df = pd.DataFrame(validation_result)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [33]:
final_merge = final_spbu_density.merge(
    validation_df,
    on=["NAME_1", "NAME_2"],
    how="left"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [34]:
final_merge["jumlah_spbu_final"] = final_merge["jumlah_spbu"]

mask = (
    (final_merge["jumlah_spbu"] == 0) &
    (final_merge["web_result_count"] > 0)
)

final_merge.loc[mask, "jumlah_spbu_final"] = 1

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [35]:
final_merge["spbu_density_final"] = (
    final_merge["jumlah_spbu_final"] /
    final_merge["luas_km2"]
) * 1000

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [36]:
spbu_density_path = f"{folder}/spbu_density_sulampua.csv"
spbu_density = pd.read_csv(spbu_density_path)


final_merge = spbu_density.merge(
    validation_df,
    on=["NAME_1", "NAME_2"],
    how="left"
)

final_merge["web_result_count"] = final_merge["web_result_count"].fillna(0)

final_merge["jumlah_spbu_final"] = final_merge["jumlah_spbu"]

mask = (
    (final_merge["jumlah_spbu"] == 0) &
    (final_merge["web_result_count"] > 0)
)

final_merge.loc[mask, "jumlah_spbu_final"] = 1

final_merge["spbu_density_final"] = (
    final_merge["jumlah_spbu_final"] /
    final_merge["luas_km2"]
) * 1000

final_merge.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,NAME_1,NAME_2,jumlah_spbu,luas_km2,spbu_density,web_result_count,jumlah_spbu_final,spbu_density_final
0,Gorontalo,Boalemo,0.0,1856.92,0.0000,0.0,0.0,0.000000
1,Gorontalo,BoneBolango,1.0,1917.76,0.5214,0.0,1.0,0.521442
2,Gorontalo,DanauLimboto,0.0,27.25,0.0000,0.0,0.0,0.000000
3,Gorontalo,Gorontalo,12.0,2147.57,5.5877,0.0,12.0,5.587711
4,Gorontalo,GorontaloUtara,1.0,1717.78,0.5821,0.0,1.0,0.582147


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [37]:
final_merge.to_csv(
    f"{folder}/spbu_density_sulampua.csv",
    index=False
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag